# AI应用基础 · 工单M1.2《特征工程优化的故障诊断系统》 — 教师参考版
第5周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())


工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/01-AI应用基础


## 参考实现 `m12`（含依赖导入）

In [2]:
import csv
import json
import sys
import numpy as np
import numpy as np  # 通用科学计算


def m12():
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score, f1_score
    from sklearn.model_selection import (GridSearchCV, cross_val_score,
                                         train_test_split)

    data = np.genfromtxt("data/pump_fault.csv", delimiter=",", skip_header=1)
    X, y = data[:, :5], data[:, 5].astype(int)

    z = (X - X.mean(0)) / X.std(0)
    mask = (np.abs(z) < 3).all(axis=1)                # 3σ清洗
    Xc, yc = X[mask], y[mask]
    print(f"清洗: {len(X)} → {len(Xc)}")

    rpm, temp, press, vib, amp = Xc.T
    Xf = np.c_[Xc, vib / amp, temp * press]           # 派生特征×2

    def ev(tag, Xa):
        Xtr, Xte, ytr, yte = train_test_split(Xa, yc, test_size=0.25,
                                              stratify=yc, random_state=42)
        m = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xtr, ytr)
        acc = accuracy_score(yte, m.predict(Xte))
        f1 = f1_score(yte, m.predict(Xte), average="macro")
        print(f"[消融] {tag:<16} acc={acc:.3f} F1={f1:.3f}")
        return acc

    b = ev("基线", Xc)
    a1 = ev("+派生特征", Xf)
    gs = GridSearchCV(RandomForestClassifier(random_state=42),
                      {"n_estimators": [50, 100, 200]}, cv=5)
    Xtr, Xte, ytr, yte = train_test_split(Xf, yc, test_size=0.25,
                                          stratify=yc, random_state=42)
    gs.fit(Xtr, ytr)
    a2 = accuracy_score(yte, gs.predict(Xte))
    print(f"[消融] +网格搜索      acc={a2:.3f} 最优参数={gs.best_params_}")
    cv = cross_val_score(gs.best_estimator_, Xf, yc, cv=5)
    print(f"5折CV: {cv.mean():.3f} ± {cv.std():.3f}")
    print(f"提升: {(a2 - b) * 100:.1f}%")
    print("决策: 树模型对归一化不敏感(矩阵中此项打低分), 故缩放不采纳。")


## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `?` | （在此处编写你的代码——依据上方注释/文档字符串） |
| `3` | TODO-3: +新特征  (提示: 对训练/测试集用同一规则构建) TODO-4: +归一化  (Min- |
|  | Max或StandardScaler, 思考树模型是否需要→写进决策) TODO-5: |
|  | +网格搜索(GridSearchCV, 只在训练集上做!) |
| `6` | TODO-6: 5折交叉验证稳定性 |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 执行与验证

In [3]:
m12()

清洗: 450 → 442
[消融] 基线               acc=1.000 F1=1.000
[消融] +派生特征            acc=1.000 F1=1.000


[消融] +网格搜索      acc=1.000 最优参数={'n_estimators': 50}
5折CV: 1.000 ± 0.000
提升: 0.0%
决策: 树模型对归一化不敏感(矩阵中此项打低分), 故缩放不采纳。
